In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from scipy import stats
from scipy.stats import spearmanr, chi2_contingency, ttest_rel
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import confusion_matrix,classification_report,f1_score,roc_auc_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.naive_bayes import BernoulliNB
from sklearn.preprocessing import StandardScaler


In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
df=pd.read_csv('/Users/siva/Downloads/digital_marketing_campaign_dataset.csv')

In [4]:
df.tail()

,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool,Conversion
7995,15995,21,Male,24849,Email,Awareness,8518.308575,0.243792,0.116773,23,9.693602,14.227794,70,13,6,7,286,IsConfid,ToolConfid,0
7996,15996,43,Female,44718,SEO,Retention,1424.613446,0.236740,0.190061,49,9.499010,3.501106,52,13,1,5,1502,IsConfid,ToolConfid,0
7997,15997,28,Female,125471,Referral,Consideration,4609.534635,0.056526,0.133826,35,2.853241,14.618323,38,16,0,3,738,IsConfid,ToolConfid,1
7998,15998,19,Female,107862,PPC,Consideration,9476.106354,0.023961,0.138386,49,1.002964,3.876623,86,1,5,7,2709,IsConfid,ToolConfid,1
7999,15999,31,Female,93002,Email,Awareness,7743.627070,0.185670,0.057228,15,6.964739,12.763660,2,18,9,9,341,IsConfid,ToolConfid,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   CustomerID           8000 non-null   int64  
 1   Age                  8000 non-null   int64  
 2   Gender               8000 non-null   object 
 3   Income               8000 non-null   int64  
 4   CampaignChannel      8000 non-null   object 
 5   CampaignType         8000 non-null   object 
 6   AdSpend              8000 non-null   float64
 7   ClickThroughRate     8000 non-null   float64
 8   ConversionRate       8000 non-null   float64
 9   WebsiteVisits        8000 non-null   int64  
 10  PagesPerVisit        8000 non-null   float64
 11  TimeOnSite           8000 non-null   float64
 12  SocialShares         8000 non-null   int64  
 13  EmailOpens           8000 non-null   int64  
 14  EmailClicks          8000 non-null   int64  
 15  PreviousPurchases    8000 non-null   i

In [6]:
#train-test split
independent = df.drop('Conversion', axis=1)
dependent = df['Conversion']

In [7]:
independent.head()

,CustomerID,Age,Gender,Income,CampaignChannel,CampaignType,AdSpend,ClickThroughRate,ConversionRate,WebsiteVisits,PagesPerVisit,TimeOnSite,SocialShares,EmailOpens,EmailClicks,PreviousPurchases,LoyaltyPoints,AdvertisingPlatform,AdvertisingTool
0,8000,56,Female,136912,Social Media,Awareness,6497.870068,0.043919,0.088031,0,2.399017,7.396803,19,6,9,4,688,IsConfid,ToolConfid
1,8001,69,Male,41760,Email,Retention,3898.668606,0.155725,0.182725,42,2.917138,5.352549,5,2,7,2,3459,IsConfid,ToolConfid
2,8002,46,Female,88456,PPC,Awareness,1546.429596,0.277490,0.076423,2,8.223619,13.794901,0,11,2,8,2337,IsConfid,ToolConfid
3,8003,32,Female,44085,PPC,Conversion,539.525936,0.137611,0.088004,47,4.540939,14.688363,89,2,2,0,2463,IsConfid,ToolConfid
4,8004,60,Female,83964,PPC,Conversion,1678.043573,0.252851,0.109940,0,2.046847,13.993370,6,6,6,8,4345,IsConfid,ToolConfid


In [13]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical variables before scaling
# Create label encoders for categorical columns
label_encoders = {}

# Identify categorical columns (assuming they contain string/object data)
categorical_columns = independent.select_dtypes(include=['object']).columns

# Encode each categorical column
indep_encoded = independent.copy()
for column in categorical_columns:
    le = LabelEncoder()
    indep_encoded[column] = le.fit_transform(independent[column])
    label_encoders[column] = le  # Store encoder for future use

# train_test_split data with encoded features
X_train, X_test, y_train, y_test = train_test_split(indep_encoded, dependent, test_size=0.2, random_state=0)

# Standard scaler - now works with numeric data only
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [15]:

param_grid={'alpha':[0.01, 0.1, 0.5, 1, 5, 10]}
grid=GridSearchCV(BernoulliNB(),param_grid,refit=True,verbose=3,n_jobs=-1,scoring='f1_weighted')
grid.fit(X_train, y_train) 

Fitting 5 folds for each of 6 candidates, totalling 30 fits


,estimator,BernoulliNB()
,param_grid,"{'alpha': [0.01, 0.1, ...]}"
,scoring,'f1_weighted'
,n_jobs,-1
,refit,True
,cv,None
,verbose,3
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,10


In [19]:
re=grid.cv_results_
grid_predictions = grid.predict(X_test_scaled) 
#confusion matrix
cm = confusion_matrix(y_test, grid_predictions)
#classification_report
clf_report = classification_report(y_test, grid_predictions)
#f1_score
f1_macro=f1_score(y_test,grid_predictions,average='weighted')
#accuracy
accuracy = accuracy_score(y_test, grid_predictions)
#roc_auc_score
roc_score = roc_auc_score(y_test,grid.predict_proba(X_test_scaled)[:,1])

In [20]:
print("The f1_macro value for best parameter {}:".format(grid.best_params_),f1_macro)
print("\nThe confusion Matrix:\n",cm)
print("\nThe Accuracy:\n",accuracy)
print("\nThe report:\n",clf_report)
print("\nROC_AUC_Score:",roc_score)

The f1_macro value for best parameter {'alpha': 10}: 0.02748725993964525

The confusion Matrix:
 [[ 194    0]
 [1405    1]]

The Accuracy:
 0.121875

The report:
               precision    recall  f1-score   support

           0       0.12      1.00      0.22       194
           1       1.00      0.00      0.00      1406

    accuracy                           0.12      1600
   macro avg       0.56      0.50      0.11      1600
weighted avg       0.89      0.12      0.03      1600


ROC_AUC_Score: 0.6767681952163775
